# Ordered Logistic Regression Results: FAIR² Dataset Exploration with `mlcroissant`

This notebook demonstrates how to load and explore the [FAIR² dataset](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json) using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

> **Dataset:** Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya

### Dataset Source
The dataset source is defined by a [Croissant schema](https://mlcommons.org/croissant/) at the following URL &ndash; all data exploration references entities by their Croissant `@id`.


In [ ]:
# Install the mlcroissant library if not already present
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata and available records from FAIR² using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Croissant schema URL for FAIR²
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Display high-level metadata
meta = dataset.metadata
print(f'Dataset name: {meta.name}\n')
print(f'Description: {meta.description}\n')
print(f'Published: {meta.datePublished} | Version: {meta.version}')
print(f'License: {meta.license}')

## 2. Data Overview
Fair² is composed of one or more **record sets** (tables). We inspect the available record sets, their fields, and underlying columns, always referencing the Croissant `@id` for each entity.

In [ ]:
# List all available record sets in the dataset using their @id
record_sets = list(dataset.record_sets)
print("Available Record Sets (by @id):")
for rs in record_sets:
    print(f"- {rs['@id']}: {rs.get('name', '')}")
    # List fields for each record set
    if 'field' in rs:
        fields = rs['field'] if isinstance(rs['field'], list) else [rs['field']]
        print("  Fields:")
        for f in fields:
            if isinstance(f, dict):
                print(f"    - {f.get('@id','')}: {f.get('name','')}")
            else:
                print(f"    - {f}")
    print()
# Set variable for the main record set's @id (choose first)
if record_sets:
    main_record_set_id = record_sets[0]['@id']
else:
    main_record_set_id = None

## 3. Data Extraction
Extract all records from one or more record sets into pandas DataFrames.

> Use record set and field `@id` values from the overview above. For demonstration, we extract the first available record set.

In [ ]:
if main_record_set_id is None:
    raise ValueError('No record set available in the dataset.')

# Prepare list of record set @ids (for extensibility)
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

for rs_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=rs_id))
        if records:
            dataframes[rs_id] = pd.DataFrame(records)
            print(f"Loaded record set '{rs_id}' with shape {dataframes[rs_id].shape}")
    except Exception as e:
        print(f"Could not load '{rs_id}': {repr(e)}")

# Preview columns of the main record set
df_main = dataframes[main_record_set_id]
print(f"\nColumns in {main_record_set_id} (by @id): \n{list(df_main.columns)}")
df_main.head()

## 4. Exploratory Data Analysis (EDA)
Process and explore data using traditional data science steps: filtering, normalization, grouping, and summary statistics.

- **Note:** Field (column) names here are referenced by their Croissant `@id` whenever possible.

In [ ]:
# Identify a numeric field for transformation (e.g., a log_likelihood or coefficient field)
numeric_fields = [col for col in df_main.columns if (df_main[col].dtype==float or df_main[col].dtype==int or pd.api.types.is_numeric_dtype(df_main[col]))]

if numeric_fields:
    numeric_field_id = numeric_fields[0]  # Use first numeric field by @id
else:
    # Select any column that seems numeric after inspection
    numeric_field_id = df_main.columns[0]

print(f"Selected numeric field (by @id): {numeric_field_id}")

# Filter records with the numeric field above a threshold
threshold = df_main[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(df_main[numeric_field_id]) else 0
filtered_df = df_main[df_main[numeric_field_id] > threshold]
print(f"Filtered records where {numeric_field_id} > {threshold:.2f}:")
print(filtered_df.head())

# Normalize the numeric field in the filtered dataframe
filtered_df = filtered_df.copy()
norm_field = f"{numeric_field_id}_normalized"
filtered_df[norm_field] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"\nNormalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, norm_field]].head())

# Attempt grouping by a likely categorical field (choose next column or a column with 'group' or 'ward' in the @id)
group_field_candidates = [col for col in df_main.columns if col != numeric_field_id and ("group" in col.lower() or "ward" in col.lower() or "type" in col.lower() or df_main[col].dtype=='object')]
group_field = group_field_candidates[0] if group_field_candidates else None
if group_field is not None:
    grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index().sort_values(numeric_field_id, ascending=False)
    print(f"\nGrouped mean {numeric_field_id} by '{group_field}' (@id):")
    print(grouped_df.head())

## 5. Visualization
Visualize the distribution of the selected numeric field and its relationship to the grouping variable (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(7,4))
sns.histplot(df_main[numeric_field_id].dropna(), bins=30, kde=True)
plt.title(f"Distribution of {numeric_field_id} (@id)")
plt.xlabel(numeric_field_id)
plt.ylabel('Count')
plt.show()

# If grouping field identified, show group differences
if group_field is not None:
    plt.figure(figsize=(10,5))
    order = filtered_df[group_field].value_counts().index
    sns.boxplot(x=group_field, y=numeric_field_id, data=filtered_df, order=order)
    plt.title(f"{numeric_field_id} by {group_field} (@id)")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion

In this notebook, we:
- Loaded and inspected metadata and records from the FAIR² dataset using `mlcroissant`;
- Explored available record sets, fields, and referenced them by their `@id` fields;
- Performed filtering, normalization, grouping, and basic visualization of a selected numeric field;
- Demonstrated a reproducible EDA workflow for data described by a Croissant schema.

> You can extend this analysis further using the loaded DataFrames and Croissant's metadata for context-aware data exploration across all data files and record sets in the package.